# Tiền xử lý dữ liệu cho bài toán dự báo PM2.5 sau 24 giờ

## Định nghĩa bài toán

Bài toán học máy là **hồi quy trực tiếp nồng độ PM2.5 tại đúng thời điểm sau 24 giờ kể từ thời điểm phát hành dự báo** cho Hà Nội, TP.HCM và Đà Nẵng. Biến mục tiêu có đơn vị ug/m^3. Đây không phải bài toán ước lượng AQI hiện tại và cũng không phải dự báo đệ quy 24 bước.

## Dữ liệu đầu vào và mục tiêu xử lý

Notebook bắt đầu từ file data/processed/all_cities.csv, gồm dữ liệu theo giờ về chất lượng không khí, thời tiết, thời gian và đặc trưng vị trí của ba thành phố Việt Nam. Mục tiêu là tạo một bảng huấn luyện chuẩn hóa, có thể tái lập và không tự tạo thêm quan sát giả.

Quy trình gồm năm bước:

1. kiểm tra schema, timestamp, thành phố và khóa trùng lặp;
2. chuyển các giá trị phi vật lý thành dữ liệu thiếu;
3. chỉ nội suy khoảng trống nội bộ không quá sáu giờ trong từng thành phố;
4. tạo đặc trưng lịch, độ trễ, cửa sổ trượt, tương tác khí tượng theo nguyên tắc nhân quả;
5. xuất file pm25_training_data_enriched.csv cùng báo cáo làm sạch dạng máy đọc được.

Mọi đặc trưng lịch sử chỉ sử dụng thông tin có tại hoặc trước thời điểm dự báo t. Nhãn tại t+24h được tạo bằng phép nối theo timestamp, tránh giả định sai rằng hai dòng liên tiếp luôn cách nhau đúng một giờ.

**Bảng 1. Các dòng đầu của dữ liệu đầu vào đã gộp; nồng độ chất ô nhiễm có đơn vị ug/m^3, đơn vị thời tiết tuân theo schema nguồn.**


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.casefold() == "model":
    PROJECT_ROOT = PROJECT_ROOT.parent

SOURCE_PATH = PROJECT_ROOT / "data" / "processed" / "all_cities.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "pm25_training_data_enriched.csv"
REPORT_PATH = PROJECT_ROOT / "model" / "results" / "data_cleaning_report.json"

VALID_RANGES = {
    "pm25": (0.0, None), "pm10": (0.0, None), "o3": (0.0, None),
    "no2": (0.0, None), "so2": (0.0, None), "co": (0.0, None),
    "temp": (-10.0, 55.0), "humidity": (0.0, 100.0),
    "wind_speed": (0.0, None), "wind_dir": (0.0, 360.0),
    "precip": (0.0, None), "pressure": (850.0, 1100.0),
    "cloud_cover": (0.0, 100.0),
}
EXPECTED_CITIES = {"H\u00e0 N\u1ed9i", "TP.HCM", "\u0110\u00e0 N\u1eb5ng"}

raw = pd.read_csv(SOURCE_PATH, low_memory=False)
print(f"Nguồn dữ liệu: {SOURCE_PATH}")
print(f"Kích thước đầu vào: {raw.shape}")
display(raw.head())

Nguồn dữ liệu: D:\Project123456\aqi-vietnam\aqi-vietnam\data\processed\all_cities.csv
Kích thước đầu vào: (98907, 34)


,datetime,pm25,pm10,o3,no2,so2,co,eu_aqi,aqi,city,...,season,pm25_lag_1h,pm25_lag_3h,pm25_lag_6h,pm25_lag_12h,pm25_lag_24h,pm25_roll_6h,pm25_roll_24h,pm25_roll_72h,aqi_category
0,2022-08-05 07:00:00,20.3,29.0,44.0,17.30,8.10,345.0,72.503334,112.822914,Hà Nội,...,Hạ,NaN,NaN,NaN,NaN,NaN,20.299999,20.299999,20.299999,Kém
1,2022-08-05 08:00:00,17.2,24.7,54.0,17.95,9.70,372.0,71.836670,110.739586,Hà Nội,...,Hạ,20.3,NaN,NaN,NaN,NaN,18.750000,18.750000,18.750000,Kém
2,2022-08-05 09:00:00,17.8,25.5,68.0,18.85,11.95,410.0,71.409996,109.406260,Hà Nội,...,Hạ,17.2,NaN,NaN,NaN,NaN,18.433333,18.433333,18.433333,Kém
3,2022-08-05 10:00:00,20.4,29.1,87.0,20.00,14.70,456.0,70.913330,107.854164,Hà Nội,...,Hạ,17.8,20.3,NaN,NaN,NaN,18.925000,18.925000,18.925000,Kém
4,2022-08-05 11:00:00,22.2,31.8,98.0,20.40,15.85,476.0,70.430000,106.343760,Hà Nội,...,Hạ,20.4,17.2,NaN,NaN,NaN,19.580000,19.580000,19.580000,Kém


## 1. Kiểm tra schema, miền giá trị vật lý và dữ liệu thiếu

Mỗi dòng phải có khóa (city, datetime) hợp lệ và đầy đủ các cột ô nhiễm, thời tiết bắt buộc. Các khóa thành phố-thời gian bị trùng sẽ làm notebook dừng thay vì âm thầm gộp dữ liệu. Giá trị số nằm ngoài miền vật lý hợp lý được chuyển thành thiếu trước khi nội suy, ví dụ nồng độ âm, độ ẩm ngoài 0-100%, hướng gió ngoài 0-360 độ hoặc áp suất ngoài 850-1100 hPa.

Nội suy chỉ áp dụng cho khoảng trống nội bộ dài tối đa sáu giờ và được thực hiện độc lập theo từng thành phố. Các đoạn biên không có quan sát đầy đủ bị cắt bỏ vì ngoại suy ra ngoài vùng đã quan sát có thể tạo xu hướng giả. Chính sách này chấp nhận giảm nhẹ độ phủ để hạn chế rò rỉ và bịa dữ liệu.

**Bảng 2. Kiểm toán làm sạch theo biến: số giá trị không hợp lệ, số giá trị được nội suy và số giá trị còn thiếu.**


In [2]:
required = {"datetime", "city", *VALID_RANGES}
missing_columns = sorted(required.difference(raw.columns))
if missing_columns:
    raise ValueError(f"Thiếu các cột bắt buộc: {missing_columns}")

cleaned = raw.copy()
cleaned["datetime"] = pd.to_datetime(cleaned["datetime"], errors="raise")
cleaned["city"] = cleaned["city"].astype(str).str.strip()
unknown_cities = sorted(set(cleaned["city"].dropna()) - EXPECTED_CITIES)
if unknown_cities:
    raise ValueError(f"Thành phố không hợp lệ: {unknown_cities}")

cleaned = cleaned.sort_values(["city", "datetime"]).reset_index(drop=True)
if cleaned.duplicated(["city", "datetime"]).any():
    raise ValueError("Có dòng trùng khóa city + datetime")

invalid_counts = {}
missing_before = {}
for column, (lower, upper) in VALID_RANGES.items():
    values = pd.to_numeric(cleaned[column], errors="coerce")
    missing_before[column] = int(values.isna().sum())
    invalid = pd.Series(False, index=cleaned.index)
    if lower is not None:
        invalid |= values < lower
    if upper is not None:
        invalid |= values > upper
    invalid_counts[column] = int(invalid.sum())
    cleaned[column] = values.mask(invalid)

numeric_columns = list(VALID_RANGES)
trimmed_parts = []
boundary_rows_trimmed = {}
for city, city_frame in cleaned.groupby("city", sort=False):
    complete = city_frame[numeric_columns].notna().all(axis=1)
    if not complete.any():
        raise ValueError(f"{city} không có dòng dữ liệu nguồn hợp lệ")
    first_valid = complete[complete].index[0]
    last_valid = complete[complete].index[-1]
    trimmed = city_frame.loc[first_valid:last_valid].copy()
    boundary_rows_trimmed[city] = int(len(city_frame) - len(trimmed))
    trimmed_parts.append(trimmed)

cleaned = pd.concat(trimmed_parts, ignore_index=True)
cleaned = cleaned.sort_values(["city", "datetime"]).reset_index(drop=True)
before_interpolation = cleaned[numeric_columns].isna().sum()
cleaned[numeric_columns] = cleaned.groupby("city", sort=False)[numeric_columns].transform(
    lambda group: group.interpolate(
        method="linear", limit=6, limit_direction="both", limit_area="inside"
    )
)
after_interpolation = cleaned[numeric_columns].isna().sum()

cleaning_report = {
    "schema_version": 1,
    "rows": int(len(cleaned)),
    "interpolation_limit_hours": 6,
    "boundary_rows_trimmed": boundary_rows_trimmed,
    "invalid_values_replaced": invalid_counts,
    "missing_before_cleaning": missing_before,
    "values_interpolated": {
        column: int(before_interpolation[column] - after_interpolation[column])
        for column in numeric_columns
    },
    "missing_after_cleaning": {
        column: int(after_interpolation[column]) for column in numeric_columns
    },
}
display(pd.DataFrame({
    "invalid": pd.Series(invalid_counts),
    "interpolated": pd.Series(cleaning_report["values_interpolated"]),
    "missing_after": pd.Series(cleaning_report["missing_after_cleaning"]),
}))

,invalid,interpolated,missing_after
pm25,0,0,0
pm10,0,0,0
o3,13,13,0
no2,0,0,0
so2,0,0,0
co,0,0,0
temp,0,0,0
humidity,0,0,0
wind_speed,0,0,0
wind_dir,0,0,0


## 2. Xây dựng đặc trưng theo nguyên tắc nhân quả

Các đặc trưng được thiết kế để biểu diễn tính dai dẳng, chu kỳ ngày/tuần, tác động khuếch tán của thời tiết và bối cảnh vị trí tĩnh.

**Bảng 3. Các nhóm đặc trưng được xây dựng và vai trò trong dự báo.**

| Nhóm đặc trưng | Ví dụ | Ý nghĩa |
|---|---|---|
| Lịch sử ô nhiễm | lag 1-168 giờ, trung bình/độ lệch chuẩn trượt, min/max 24 giờ | biểu diễn tính dai dẳng, lặp lại theo ngày và biến động gần đây |
| Lịch sử cùng giờ | mean/median cùng giờ trong bảy ngày | biểu diễn quy luật ngày mà không dùng dữ liệu tương lai |
| Chu kỳ thời gian | sin/cos của giờ, ngày và tháng | mã hóa chu kỳ mà không tạo điểm gián đoạn giả |
| Khí tượng | nhiệt độ, độ ẩm, gió, mưa, áp suất | biểu diễn khuếch tán, ứ đọng và rửa trôi ô nhiễm |
| Tương tác | chỉ số thông gió, ứ đọng ẩm, tỷ lệ chất ô nhiễm | đưa quan hệ phi tuyến có ý nghĩa vật lý vào dữ liệu |

Cửa sổ trượt có thể sử dụng quan sát tại thời điểm phát hành t vì đây là thông tin hợp lệ khi dự báo. Không sử dụng cửa sổ căn giữa, độ trễ âm hoặc quan sát thời tiết tương lai.


In [3]:
def safe_ratio(numerator, denominator, default=0.0):
    numerator = pd.Series(numerator, copy=False).astype(float)
    denominator = pd.Series(denominator, copy=False).astype(float).replace(0.0, np.nan)
    return numerator.div(denominator).replace([np.inf, -np.inf], np.nan).fillna(default)


def enrich_city_history(city_df):
    city_df = city_df.sort_values("datetime").copy()
    for lag in (1, 3, 6, 12, 24, 48, 72, 96, 120, 144, 168):
        city_df[f"pm25_lag_{lag}h"] = city_df["pm25"].shift(lag)

    for window in (6, 12, 24, 72, 168):
        rolling = city_df["pm25"].rolling(window=window, min_periods=1)
        city_df[f"pm25_roll_{window}h"] = rolling.mean()
        city_df[f"pm25_std_{window}h"] = rolling.std().fillna(0.0)

    city_df["pm25_min_24h"] = city_df["pm25"].rolling(24, min_periods=1).min()
    city_df["pm25_max_24h"] = city_df["pm25"].rolling(24, min_periods=1).max()
    city_df["pm25_delta_1h"] = city_df["pm25"] - city_df["pm25_lag_1h"]
    city_df["pm25_delta_3h"] = city_df["pm25"] - city_df["pm25_lag_3h"]
    city_df["pm25_delta_24h"] = city_df["pm25"] - city_df["pm25_lag_24h"]
    city_df["pm25_roll_ratio_6h_24h"] = safe_ratio(city_df["pm25_roll_6h"], city_df["pm25_roll_24h"])
    city_df["pm25_roll_ratio_24h_72h"] = safe_ratio(city_df["pm25_roll_24h"], city_df["pm25_roll_72h"])

    same_hour_columns = ["pm25", *[f"pm25_lag_{lag}h" for lag in (24, 48, 72, 96, 120, 144)]]
    same_hour = city_df[same_hour_columns]
    city_df["pm25_same_hour_mean_7d"] = same_hour.mean(axis=1)
    city_df["pm25_same_hour_median_7d"] = same_hour.median(axis=1)
    city_df["pm25_same_hour_std_7d"] = same_hour.std(axis=1).fillna(0.0)
    city_df["pm25_same_hour_min_7d"] = same_hour.min(axis=1)
    city_df["pm25_same_hour_max_7d"] = same_hour.max(axis=1)
    city_df["pm25_same_hour_ratio_7d"] = safe_ratio(city_df["pm25"], city_df["pm25_same_hour_mean_7d"])
    city_df["pm25_weekly_delta"] = city_df["pm25"] - city_df["pm25_lag_168h"]
    return city_df


enriched = cleaned.copy()
enriched["year"] = enriched["datetime"].dt.year
enriched["month"] = enriched["datetime"].dt.month
enriched["day"] = enriched["datetime"].dt.day
enriched["hour"] = enriched["datetime"].dt.hour
enriched["day_of_week"] = enriched["datetime"].dt.dayofweek
enriched["is_weekend"] = enriched["day_of_week"].isin([5, 6]).astype(int)
enriched["day_of_year"] = enriched["datetime"].dt.dayofyear
enriched["season"] = enriched["month"].map({12:"\u0110\u00f4ng",1:"\u0110\u00f4ng",2:"\u0110\u00f4ng",3:"Xu\u00e2n",4:"Xu\u00e2n",5:"Xu\u00e2n",6:"H\u1ea1",7:"H\u1ea1",8:"H\u1ea1",9:"Thu",10:"Thu",11:"Thu"})
enriched = pd.concat(
    [enrich_city_history(group) for _, group in enriched.groupby("city", sort=False)],
    ignore_index=True,
)

wind_radians = np.deg2rad(enriched["wind_dir"].astype(float).fillna(0.0))
enriched["wind_x"] = np.sin(wind_radians)
enriched["wind_y"] = np.cos(wind_radians)
enriched["ventilation_index"] = enriched["wind_speed"] * (100.0 - enriched["humidity"]) / 100.0
enriched["humid_stagnation"] = enriched["humidity"] / (enriched["wind_speed"] + 1.0)
enriched["rain_flag"] = (enriched["precip"] > 0.0).astype(int)
enriched["calm_wind"] = (enriched["wind_speed"] < 2.0).astype(int)
enriched["high_humidity"] = (enriched["humidity"] > 85.0).astype(int)
enriched["hour_sin"] = np.sin(2.0 * np.pi * enriched["hour"] / 24.0)
enriched["hour_cos"] = np.cos(2.0 * np.pi * enriched["hour"] / 24.0)
enriched["day_sin"] = np.sin(2.0 * np.pi * enriched["day_of_year"] / 365.25)
enriched["day_cos"] = np.cos(2.0 * np.pi * enriched["day_of_year"] / 365.25)
enriched["month_sin"] = np.sin(2.0 * np.pi * enriched["month"] / 12.0)
enriched["month_cos"] = np.cos(2.0 * np.pi * enriched["month"] / 12.0)
enriched["pm25_pm10_ratio"] = safe_ratio(enriched["pm25"], enriched["pm10"])
enriched["no2_co_ratio"] = safe_ratio(enriched["no2"], enriched["co"])

enriched = enriched.replace([np.inf, -np.inf], np.nan)
print(f"Kích thước sau khi tạo đặc trưng: {enriched.shape}")

Kích thước sau khi tạo đặc trưng: (98907, 76)


## 3. Kiểm tra đầu ra và xuất dữ liệu

Trước khi xuất file, notebook xác nhận:

- số dòng không thay đổi sau khi tạo đặc trưng;
- khóa (city, datetime) vẫn duy nhất;
- có đủ ba thành phố yêu cầu;
- timestamp tăng đơn điệu trong từng thành phố;
- các phép đo số cốt lõi không còn giá trị thiếu;
- trên 99% thời điểm phát hành có quan sát đúng 24 giờ sau.

Điều kiện cuối được kiểm tra bằng phép nối timestamp thay vì shift(-24). Phân biệt này rất quan trọng vì dịch theo số dòng sẽ sai nếu chuỗi bị thiếu một timestamp theo giờ. Dữ liệu chỉ được ghi ra file sau khi toàn bộ assertion vượt qua.


In [4]:
assert len(enriched) == len(cleaned)
assert enriched.duplicated(["city", "datetime"]).sum() == 0
assert set(enriched["city"].unique()) == EXPECTED_CITIES
assert enriched["datetime"].is_monotonic_increasing is False  # dữ liệu được sắp xếp theo thành phố rồi theo thời gian
assert enriched.groupby("city")["datetime"].apply(lambda s: s.is_monotonic_increasing).all()
assert enriched[numeric_columns].isna().sum().sum() == 0

# Verify the exact t+24h target using a timestamp join, not a row shift.
future = enriched[["city", "datetime", "pm25"]].rename(
    columns={"datetime": "target_time", "pm25": "target_pm25_24h"}
)
coverage_probe = enriched[["city", "datetime"]].copy()
coverage_probe["target_time"] = coverage_probe["datetime"] + pd.Timedelta(hours=24)
coverage_probe = coverage_probe.merge(future, on=["city", "target_time"], how="left")
target_coverage = float(coverage_probe["target_pm25_24h"].notna().mean())
assert target_coverage > 0.99

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
enriched.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
REPORT_PATH.write_text(json.dumps(cleaning_report, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Đã lưu: {OUTPUT_PATH}")
print(f"Số dòng: {len(enriched):,}; số cột: {len(enriched.columns)}")
print(f"Khoảng thời gian: {enriched['datetime'].min()} -> {enriched['datetime'].max()}")
print(f"Độ phủ nhãn chính xác tại t+24h: {target_coverage:.4%}")

Đã lưu: D:\Project123456\aqi-vietnam\aqi-vietnam\data\processed\pm25_training_data_enriched.csv
Số dòng: 98,907; số cột: 76
Khoảng thời gian: 2022-08-05 07:00:00 -> 2026-05-09 23:00:00
Độ phủ nhãn chính xác tại t+24h: 99.9272%


## 4. Thống kê mô tả sau tiền xử lý

Bảng thống kê theo thành phố là bước kiểm tra hợp lý cuối cùng. Trung bình, độ lệch chuẩn và cực đại PM2.5 cho biết thành phố nào có phân phối khó dự báo hơn. Vì vậy, giai đoạn đánh giá phải báo cáo chỉ số theo từng thành phố thay vì chỉ dùng một điểm tổng hợp.

**Bảng 4. Số quan sát và phân phối PM2.5 theo thành phố; các thống kê PM2.5 có đơn vị ug/m^3.**

**Bảng 5. Thống kê mô tả các biến ô nhiễm và thời tiết chính; đơn vị tuân theo schema nguồn.**


In [5]:
summary = enriched.groupby("city").agg(
    rows=("pm25", "size"),
    start=("datetime", "min"),
    end=("datetime", "max"),
    pm25_mean=("pm25", "mean"),
    pm25_std=("pm25", "std"),
    pm25_max=("pm25", "max"),
).reset_index()
display(summary)
display(enriched[["pm25", "pm10", "temp", "humidity", "wind_speed"]].describe().T)

,city,rows,start,end,pm25_mean,pm25_std,pm25_max
0,Hà Nội,32969,2022-08-05 07:00:00,2026-05-09 23:00:00,45.190094,27.642812,232.8
1,TP.HCM,32969,2022-08-05 07:00:00,2026-05-09 23:00:00,26.003716,13.736189,120.6
2,Đà Nẵng,32969,2022-08-05 07:00:00,2026-05-09 23:00:00,18.993242,10.455467,149.9


,count,mean,std,min,25%,50%,75%,max
pm25,98907.0,30.062350,21.832340,0.300000,15.900000,23.900000,36.800000,232.800000
pm10,98907.0,39.524225,26.427895,0.400000,22.000000,32.700000,48.300000,291.500000
temp,98907.0,25.982529,4.410119,6.500000,23.900000,26.150000,28.650000,41.150000
humidity,98907.0,79.620598,14.221492,21.568388,71.399284,83.052510,90.936750,100.000000
wind_speed,98907.0,8.864438,5.126639,0.000000,5.091168,7.993297,11.753877,60.716324


## Kết luận và hạn chế dữ liệu

Notebook tái tạo bảng dữ liệu huấn luyện chuẩn hoàn toàn từ các quan sát đã crawl và gộp. Quy trình ghi nhận cách xử lý giá trị không hợp lệ, giới hạn nội suy ở khoảng trống nội bộ ngắn và chỉ xây dựng các biến dự báo theo nguyên tắc nhân quả.

Dữ liệu vẫn có các hạn chế quan trọng. Sản phẩm khí quyển dạng lưới có thể làm trơn các đỉnh ô nhiễm ở quy mô khu phố; ba chuỗi cấp thành phố không thể mô tả toàn bộ khác biệt cấp quận. Những hạn chế này có thể làm giảm khả năng dự báo các sự kiện PM2.5 cực đoan và phải được xem xét trong phần phân tích lỗi. Không áp dụng tăng cường dữ liệu vì chuỗi tổng hợp có thể làm sai lệch các đợt ô nhiễm thực tế.
